In [1]:
import numpy as np

In [3]:
# dataset_generator.py
# Usage:  python dataset_generator.py --sessions 100000 --transactions 25000 --out data
import json
import random
import datetime
import uuid
import threading
import numpy as np
from faker import Faker

fake = Faker()

# --- Configuration (override from the command line) ---
import argparse
import os

parser = argparse.ArgumentParser(description="Generate the synthetic e-commerce dataset.")
parser.add_argument("--users", type=int, default=10000)
parser.add_argument("--products", type=int, default=5000)
parser.add_argument("--categories", type=int, default=25)
parser.add_argument("--sessions", type=int, default=100000,
                    help="Original brief uses 2,000,000. Default is dev-sized.")
parser.add_argument("--transactions", type=int, default=25000,
                    help="Original brief uses 500,000. Default is dev-sized.")
parser.add_argument("--days", type=int, default=90)
parser.add_argument("--chunk-size", type=int, default=25000,
                    help="Sessions per sessions_N.json file.")
parser.add_argument("--out", type=str, default="data", help="Output folder.")
args, _unknown = parser.parse_known_args()  # ignore extra args (e.g. Jupyter passes -f kernel.json)

NUM_USERS = args.users
NUM_PRODUCTS = args.products
NUM_CATEGORIES = args.categories
NUM_TRANSACTIONS = args.transactions
NUM_SESSIONS = args.sessions
TIMESPAN_DAYS = args.days
MAX_ITERATIONS = (NUM_SESSIONS + NUM_TRANSACTIONS) * 2  # Fail-safe
OUT_DIR = args.out
os.makedirs(OUT_DIR, exist_ok=True)

# --- Initialization ---
np.random.seed(42)
random.seed(42)
Faker.seed(42)

print("Initializing dataset generation...")

# --- ID Generators ---
def generate_session_id():
    return f"sess_{uuid.uuid4().hex[:10]}"

def generate_transaction_id():
    return f"txn_{uuid.uuid4().hex[:12]}"

# --- Inventory Management ---
class InventoryManager:
    def __init__(self, products):
        self.products = {p["product_id"]: p for p in products}
        self.lock = threading.RLock()  # For thread safety

    def update_stock(self, product_id, quantity):
        with self.lock:
            if product_id not in self.products:
                return False
            if self.products[product_id]["current_stock"] >= quantity:
                self.products[product_id]["current_stock"] -= quantity
                return True
            return False

    def get_product(self, product_id):
        with self.lock:
            return self.products.get(product_id)

# --- Helper Functions ---
def determine_page_type(position, previous_pages):
    """
    Determine page type based on position in user journey and previous pages viewed.
    """
    if position == 0:
        # First page is usually homepage, search or category
        return random.choice(["home", "search", "category_listing"])

    if not previous_pages:
        return "home"

    prev_page = previous_pages[-1]["page_type"]

    # Define realistic page flow transitions
    if prev_page == "home":
        return random.choices(
            ["category_listing", "search", "product_detail"],
            weights=[0.5, 0.3, 0.2]
        )[0]
    elif prev_page == "category_listing":
        return random.choices(
            ["product_detail", "category_listing", "search", "home"],
            weights=[0.7, 0.1, 0.1, 0.1]
        )[0]
    elif prev_page == "search":
        return random.choices(
            ["product_detail", "search", "category_listing", "home"],
            weights=[0.6, 0.2, 0.1, 0.1]
        )[0]
    elif prev_page == "product_detail":
        return random.choices(
            ["product_detail", "cart", "category_listing", "search", "home"],
            weights=[0.3, 0.3, 0.2, 0.1, 0.1]
        )[0]
    elif prev_page == "cart":
        return random.choices(
            ["checkout", "product_detail", "category_listing", "home"],
            weights=[0.6, 0.2, 0.1, 0.1]
        )[0]
    elif prev_page == "checkout":
        return random.choices(
            ["confirmation", "cart", "home"],
            weights=[0.8, 0.1, 0.1]
        )[0]
    elif prev_page == "confirmation":
        return random.choices(
            ["home", "product_detail", "category_listing"],
            weights=[0.6, 0.2, 0.2]
        )[0]
    else:
        return "home"

def get_page_content(page_type, products_list, categories_list, inventory):
    """
    Get appropriate product and category based on page type.
    Returns active products and categories with stock.
    """
    if page_type == "product_detail":
        # Try to find an active product with stock
        attempts = 0
        while attempts < 10:  # Limit attempts to avoid infinite loop
            product = random.choice(products_list)
            if product["is_active"] and product["current_stock"] > 0:
                category_id = product["category_id"]
                category = next((c for c in categories_list if c["category_id"] == category_id), None)
                return product, category
            attempts += 1

        # If we couldn't find a suitable product after attempts, just return any
        product = random.choice(products_list)
        category_id = product["category_id"]
        category = next((c for c in categories_list if c["category_id"] == category_id), None)
        return product, category

    elif page_type == "category_listing":
        category = random.choice(categories_list)
        return None, category

    else:
        return None, None

# --- Category Generation ---
categories = []
for cat_id in range(NUM_CATEGORIES):
    category = {
        "category_id": f"cat_{cat_id:03d}",
        "name": fake.company(),
        "subcategories": []
    }

    for sub_id in range(random.randint(3, 5)):
        subcategory = {
            "subcategory_id": f"sub_{cat_id:03d}_{sub_id:02d}",
            "name": fake.bs(),
            "profit_margin": round(random.uniform(0.1, 0.4), 2)
        }
        category["subcategories"].append(subcategory)

    categories.append(category)

print(f"Generated {len(categories)} categories")

# --- Product Generation ---
products = []
product_creation_start = datetime.datetime.now() - datetime.timedelta(days=TIMESPAN_DAYS*2)

for prod_id in range(NUM_PRODUCTS):
    category = random.choice(categories)

    # Generate price history with 1-3 price points
    base_price = round(random.uniform(5, 500), 2)
    price_history = []

    # Initial price
    initial_date = fake.date_time_between(
        start_date=product_creation_start,
        end_date=product_creation_start + datetime.timedelta(days=TIMESPAN_DAYS//3)
    )
    price_history.append({
        "price": base_price,
        "date": initial_date.isoformat()
    })

    # Add 0-2 more price changes
    for _ in range(random.randint(0, 2)):
        price_change_date = fake.date_time_between(
            start_date=initial_date,
            end_date="now"
        )
        new_price = round(base_price * random.uniform(0.8, 1.2), 2)  # +/- 20%
        price_history.append({
            "price": new_price,
            "date": price_change_date.isoformat()
        })
        initial_date = price_change_date

    # Sort price history by date
    price_history.sort(key=lambda x: x["date"])

    # Get current price (most recent in history)
    current_price = price_history[-1]["price"]

    products.append({
        "product_id": f"prod_{prod_id:05d}",
        "name": fake.catch_phrase().title(),
        "category_id": category["category_id"],
        "base_price": current_price,
        "current_stock": random.randint(10, 1000),  # Minimum stock
        "is_active": random.choices([True, False], weights=[0.95, 0.05])[0],
        "price_history": price_history,
        "creation_date": price_history[0]["date"]
    })

print(f"Generated {len(products)} products")

# --- User Generation ---
users = []
for user_id in range(NUM_USERS):
    reg_date = fake.date_time_between(
        start_date=f"-{TIMESPAN_DAYS*3}d",
        end_date=f"-{TIMESPAN_DAYS}d"
    )

    users.append({
        "user_id": f"user_{user_id:06d}",
        "geo_data": {
            "city": fake.city(),
            "state": fake.state_abbr(),
            "country": fake.country_code()
        },
        "registration_date": reg_date.isoformat(),
        "last_active": fake.date_time_between(start_date=reg_date, end_date="now").isoformat()
    })

print(f"Generated {len(users)} users")

# --- Session & Transaction Generation ---
inventory = InventoryManager(products)
sessions = []
transactions = []
transaction_counter = 0
session_counter = 0
iteration = 0

print("Generating sessions and transactions...")
while (session_counter < NUM_SESSIONS or transaction_counter < NUM_TRANSACTIONS) and iteration < MAX_ITERATIONS:
    iteration += 1

    # Session Generation
    if session_counter < NUM_SESSIONS:
        user = random.choice(users)
        session_id = generate_session_id()
        session_start = fake.date_time_between(
            start_date=f"-{TIMESPAN_DAYS}d",
            end_date="now"
        )
        session_duration = random.randint(30, 3600)  # 30 sec to 1 hour

        # Generate realistic page flow
        page_views = []
        viewed_products = set()
        cart_contents = {}

        # Generate page view timeline with proper time distribution
        time_slots = sorted([0] + [random.randint(1, session_duration-1) for _ in range(random.randint(3, 15))] + [session_duration])

        for i in range(len(time_slots)-1):
            view_duration = time_slots[i+1] - time_slots[i]
            page_type = determine_page_type(i, page_views)
            product, category = get_page_content(page_type, products, categories, inventory)

            # Track product views and cart interactions
            if page_type == "product_detail" and product:
                product_id = product["product_id"]
                viewed_products.add(product_id)

                # Only allow adding to cart for products that were viewed
                if random.random() < 0.3:
                    if product_id not in cart_contents:
                        cart_contents[product_id] = {
                            "quantity": 0,
                            "price": product["base_price"]
                        }

                    # Check current stock via inventory manager
                    max_possible = min(3, inventory.get_product(product_id)["current_stock"] - cart_contents[product_id]["quantity"])
                    if max_possible > 0:
                        add_qty = random.randint(1, max_possible)
                        cart_contents[product_id]["quantity"] += add_qty

            page_views.append({
                "timestamp": (session_start + datetime.timedelta(seconds=time_slots[i])).isoformat(),
                "page_type": page_type,
                "product_id": product["product_id"] if product else None,
                "category_id": category["category_id"] if category else None,
                "view_duration": view_duration
            })

        # Determine conversion - only if they have cart contents and viewed checkout or confirmation pages
        converted = False
        if cart_contents and any(p["page_type"] in ["checkout", "confirmation"] for p in page_views):
            converted = random.random() < 0.7  # 70% chance of completing checkout if reached checkout page

        # Geographic consistency - use user's geo plus random IP
        session_geo = user["geo_data"].copy()
        session_geo["ip_address"] = fake.ipv4()

        # Build session
        sessions.append({
            "session_id": session_id,
            "user_id": user["user_id"],
            "start_time": session_start.isoformat(),
            "end_time": (session_start + datetime.timedelta(seconds=session_duration)).isoformat(),
            "duration_seconds": session_duration,
            "geo_data": session_geo,
            "device_profile": {
                "type": random.choice(["mobile", "desktop", "tablet"]),
                "os": random.choice(["iOS", "Android", "Windows", "macOS"]),
                "browser": random.choice(["Chrome", "Safari", "Firefox", "Edge"])
            },
            "viewed_products": list(viewed_products),
            "page_views": page_views,
            "cart_contents": {k:v for k,v in cart_contents.items() if v["quantity"] > 0},
            "conversion_status": "converted" if converted else "abandoned" if cart_contents else "browsed",
            "referrer": random.choice(["direct", "email", "social", "search_engine", "affiliate"])
        })

        session_counter += 1

        # Create transaction if converted
        if converted and transaction_counter < NUM_TRANSACTIONS:
            transaction_items = []
            valid = True

            # Process each item in cart
            for prod_id, details in cart_contents.items():
                quantity = details["quantity"]
                if quantity > 0:
                    # Attempt to update inventory
                    if inventory.update_stock(prod_id, quantity):
                        transaction_items.append({
                            "product_id": prod_id,
                            "quantity": quantity,
                            "unit_price": details["price"],
                            "subtotal": round(quantity * details["price"], 2)
                        })
                    else:
                        # If any item's inventory update fails, mark transaction as invalid
                        valid = False
                        break

            if valid and transaction_items:
                # Calculate total with possible discount
                subtotal = sum(item["subtotal"] for item in transaction_items)
                discount = 0
                if random.random() < 0.2:  # 20% chance of discount
                    discount_rate = random.choice([0.05, 0.1, 0.15, 0.2])
                    discount = round(subtotal * discount_rate, 2)

                total = round(subtotal - discount, 2)

                transactions.append({
                    "transaction_id": generate_transaction_id(),
                    "session_id": session_id,  # Link to the session
                    "user_id": user["user_id"],
                    "timestamp": (session_start + datetime.timedelta(seconds=session_duration)).isoformat(),
                    "items": transaction_items,
                    "subtotal": subtotal,
                    "discount": discount,
                    "total": total,
                    "payment_method": random.choice(["credit_card", "paypal", "apple_pay", "crypto"]),
                    "status": "completed"
                })
                transaction_counter += 1

    # Generate additional transactions if needed
    if transaction_counter < NUM_TRANSACTIONS and random.random() < 0.2:
        user = random.choice(users)
        products_in_txn = random.sample(products, k=min(3, len(products)))

        transaction_items = []
        for product in products_in_txn:
            if product["is_active"]:
                quantity = random.randint(1, 3)
                if inventory.update_stock(product["product_id"], quantity):
                    transaction_items.append({
                        "product_id": product["product_id"],
                        "quantity": quantity,
                        "unit_price": product["base_price"],
                        "subtotal": round(quantity * product["base_price"], 2)
                    })

        if transaction_items:
            # Calculate total with possible discount
            subtotal = sum(item["subtotal"] for item in transaction_items)
            discount = 0
            if random.random() < 0.2:
                discount_rate = random.choice([0.05, 0.1, 0.15, 0.2])
                discount = round(subtotal * discount_rate, 2)

            total = round(subtotal - discount, 2)

            transactions.append({
                "transaction_id": generate_transaction_id(),
                "session_id": None,  # Not linked to a specific session
                "user_id": user["user_id"],
                "timestamp": fake.date_time_between(
                    start_date=f"-{TIMESPAN_DAYS}d",
                    end_date="now"
                ).isoformat(),
                "items": transaction_items,
                "subtotal": subtotal,
                "discount": discount,
                "total": total,
                "payment_method": random.choice(["credit_card", "paypal", "bank_transfer", "gift_card"]),
                "status": random.choice(["completed", "processing", "shipped", "delivered"])
            })
            transaction_counter += 1

    # Progress update
    if iteration % 10000 == 0:
        print(f"Progress: {session_counter:,}/{NUM_SESSIONS:,} sessions, {transaction_counter:,}/{NUM_TRANSACTIONS:,} transactions (iteration {iteration:,})")

# --- Data Export ---
def json_serializer(obj):
    """Custom JSON serializer for objects not serializable by default json code"""
    if isinstance(obj, (datetime.datetime, datetime.date)):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

print("Saving datasets...")

# Save users
with open(os.path.join(OUT_DIR, "users.json"), "w", encoding="utf-8") as f:
    json.dump(users, f, default=json_serializer)

# Save products with updated stock levels
with open(os.path.join(OUT_DIR, "products.json"), "w", encoding="utf-8") as f:
    json.dump(list(inventory.products.values()), f, default=json_serializer)

# Save categories
with open(os.path.join(OUT_DIR, "categories.json"), "w", encoding="utf-8") as f:
    json.dump(categories, f, default=json_serializer)

# Save transactions
with open(os.path.join(OUT_DIR, "transactions.json"), "w", encoding="utf-8") as f:
    json.dump(transactions, f, default=json_serializer)

# Save sessions in chunks
CHUNK_SIZE = args.chunk_size
for i in range(0, len(sessions), CHUNK_SIZE):
    chunk = sessions[i:i+CHUNK_SIZE]
    with open(os.path.join(OUT_DIR, f"sessions_{i//CHUNK_SIZE}.json"), "w", encoding="utf-8") as f:
        json.dump(chunk, f, default=json_serializer)

print(f"""
Dataset generation complete!
- Sessions: {len(sessions):,} (target: {NUM_SESSIONS:,})
- Transactions: {len(transactions):,} (target: {NUM_TRANSACTIONS:,})
- Remaining products: {sum(p['current_stock'] for p in inventory.products.values()):,}
""")

Initializing dataset generation...
Generated 25 categories
Generated 5000 products
Generated 10000 users
Generating sessions and transactions...
Progress: 10,000/100,000 sessions, 3,959/25,000 transactions (iteration 10,000)
Progress: 20,000/100,000 sessions, 8,038/25,000 transactions (iteration 20,000)
Progress: 30,000/100,000 sessions, 12,054/25,000 transactions (iteration 30,000)
Progress: 40,000/100,000 sessions, 16,123/25,000 transactions (iteration 40,000)
Progress: 50,000/100,000 sessions, 20,190/25,000 transactions (iteration 50,000)
Progress: 60,000/100,000 sessions, 24,268/25,000 transactions (iteration 60,000)
Progress: 70,000/100,000 sessions, 25,000/25,000 transactions (iteration 70,000)
Progress: 80,000/100,000 sessions, 25,000/25,000 transactions (iteration 80,000)
Progress: 90,000/100,000 sessions, 25,000/25,000 transactions (iteration 90,000)
Progress: 100,000/100,000 sessions, 25,000/25,000 transactions (iteration 100,000)
Saving datasets...

Dataset generation comple

In [1]:
import argparse
import os

import matplotlib
matplotlib.use("Agg")            # no window needed: just write PNG files
import matplotlib.pyplot as plt
import pandas as pd

BLUE, ORANGE, GREEN, GREY = "#2F6DB5", "#E07B39", "#3A9A6B", "#8A8F98"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 12})


def money(x, _=None):
    return f"${x / 1e6:.1f}M" if abs(x) >= 1e6 else f"${x / 1e3:.0f}K" if abs(x) >= 1e3 else f"${x:.0f}"


def pretty(label):
    """'2_one_time' -> 'One time'"""
    parts = str(label).split("_", 1)
    text = parts[1] if parts[0].isdigit() and len(parts) > 1 else str(label)
    return text.replace("_", " ").capitalize()


def read(results, name):
    path = os.path.join(results, name)
    if not os.path.exists(path):
        print(f"  skipped (missing {path})")
        return None
    return pd.read_csv(path)


def save(fig, out, name):
    os.makedirs(out, exist_ok=True)
    path = os.path.join(out, f"{name}.png")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    print(f"  wrote {path}")


# ------------------------------------------------------------------------------------------
def chart_revenue_by_category(results, out):
    df = read(results, "mongo_revenue_by_category.csv")
    if df is None:
        return
    top = df.sort_values("revenue", ascending=False).head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    bars = ax.barh(top["category_name"], top["revenue"], color=BLUE)
    ax.bar_label(bars, labels=[money(v) for v in top["revenue"]], padding=3, fontsize=9)
    ax.xaxis.set_major_formatter(money)
    ax.set_xlim(0, top["revenue"].max() * 1.15)
    ax.set_title("Revenue by product category (top 10)")
    ax.set_xlabel("Revenue from completed, shipped and delivered orders")
    share = top["revenue"].sum() / df["revenue"].sum() * 100
    ax.text(0.99, 0.02, f"Top 10 = {share:.0f}% of all revenue", transform=ax.transAxes,
            ha="right", color=GREY, fontsize=9)
    save(fig, out, "1_revenue_by_category")


def chart_monthly_revenue(results, out):
    df = read(results, "mongo_monthly_revenue.csv")
    if df is None:
        return
    df = df.sort_values("month")
    cutoff = df["orders"].median() * 0.25           # months with far fewer orders = partial months
    partial = df[df["orders"] < cutoff]
    full = df[df["orders"] >= cutoff]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(full["month"], full["revenue"], color=BLUE, label="Revenue", width=0.55)
    ax.set_ylim(0, full["revenue"].max() * 1.3)
    ax.yaxis.set_major_formatter(money)
    ax.set_ylabel("Revenue")
    ax2 = ax.twinx()
    ax2.plot(full["month"], full["orders"], color=ORANGE, marker="o", label="Orders")
    ax2.set_ylabel("Orders")
    ax2.spines["right"].set_visible(True)
    ax2.set_ylim(0, full["orders"].max() * 1.3)
    ax.set_title("Monthly revenue and order count")
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper left", frameon=False)
    if len(partial):
        ax.text(0.99, -0.16, "Partial months excluded: " + ", ".join(partial["month"]),
                transform=ax.transAxes, ha="right", color=GREY, fontsize=8)
    save(fig, out, "2_monthly_revenue")


def chart_customer_segments(results, out):
    df = read(results, "mongo_user_segmentation.csv")
    if df is None:
        return
    df = df.sort_values("segment")
    labels = [pretty(s) for s in df["segment"]]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.2))
    b1 = a1.bar(labels, df["users"], color=BLUE)
    a1.bar_label(b1, labels=[f"{v:,}" for v in df["users"]], padding=2, fontsize=9)
    a1.set_title("Users per purchase-frequency segment")
    a1.set_ylabel("Users")
    b2 = a2.bar(labels, df["avg_spend_per_user"], color=GREEN)
    a2.bar_label(b2, labels=[money(v) for v in df["avg_spend_per_user"]], padding=2, fontsize=9)
    a2.yaxis.set_major_formatter(money)
    a2.set_title("Average spend per user")
    for a in (a1, a2):
        a.tick_params(axis="x", rotation=25)
    save(fig, out, "3_customer_segments")


def chart_funnel(results, out):
    df = read(results, "integrated_funnel_overall.csv")
    if df is None:
        return
    r = df.iloc[0]
    stages = [("All sessions", r["sessions"]), ("Viewed a product", r["viewed_product"]),
              ("Reached cart", r["reached_cart"]), ("Reached checkout", r["reached_checkout"]),
              ("Placed an order", r["purchased"])]
    names = [s[0] for s in stages][::-1]
    vals = [s[1] for s in stages][::-1]
    total = stages[0][1]
    fig, ax = plt.subplots(figsize=(8, 4.2))
    bars = ax.barh(names, vals, color=[ORANGE, BLUE, BLUE, BLUE, GREEN])   # bottom -> top
    for b, v in zip(bars, vals):
        ax.text(v + total * 0.01, b.get_y() + b.get_height() / 2,
                f"{int(v):,}  ({v / total * 100:.1f}% of sessions)", va="center", fontsize=9)
    ax.set_xlim(0, total * 1.38)
    ax.set_title("Conversion funnel: browsing (HBase) to orders (MongoDB)")
    ax.set_xlabel("Sessions")
    save(fig, out, "4_conversion_funnel")


def chart_conversion_by_segment(results, out):
    dev = read(results, "integrated_funnel_by_device.csv")
    ref = read(results, "integrated_funnel_by_referrer.csv")
    if dev is None or ref is None:
        return
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
    for ax, df, key, title in ((axes[0], dev, "device_type", "By device"),
                               (axes[1], ref, "referrer", "By traffic source")):
        df = df.sort_values("session_conversion_pct", ascending=False)
        bars = ax.bar(df[key].str.replace("_", " "), df["session_conversion_pct"], color=ORANGE)
        ax.bar_label(bars, labels=[f"{v:.1f}%" for v in df["session_conversion_pct"]],
                     padding=2, fontsize=9)
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=20)
    axes[0].set_ylabel("Sessions that ended in an order (%)")
    axes[0].set_ylim(0, max(dev["session_conversion_pct"].max(),
                            ref["session_conversion_pct"].max()) * 1.25)
    fig.suptitle("Session conversion rate", fontweight="bold")
    save(fig, out, "5_conversion_by_segment")


def chart_top_products(results, out):
    df = read(results, "mongo_top_products.csv")
    if df is None:
        return
    top = df.sort_values("revenue", ascending=False).head(10).iloc[::-1]
    labels = [f"{n[:28]}…" if len(str(n)) > 28 else str(n) for n in top["name"]]
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    bars = ax.barh(labels, top["revenue"], color=BLUE)
    ax.bar_label(bars, labels=[f"{money(r)} · {u} units" for r, u in zip(top["revenue"], top["units_sold"])],
                 padding=3, fontsize=8)
    ax.xaxis.set_major_formatter(money)
    ax.set_xlim(0, top["revenue"].max() * 1.35)
    ax.set_title("Top 10 products by revenue")
    save(fig, out, "6_top_products")


def chart_cohort_heatmap(results, out):
    df = read(results, "spark_cohort.csv")
    if df is None:
        return
    pv = df.pivot(index="registration_month", columns="month_offset",
                  values="revenue_per_cohort_user").sort_index()
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.28 * len(pv) + 1.5)))
    im = ax.imshow(pv.values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(len(pv.columns)))
    ax.set_xticklabels(pv.columns)
    ax.set_yticks(range(len(pv.index)))
    ax.set_yticklabels(pv.index, fontsize=8)
    ax.set_xlabel("Months since registration")
    ax.set_ylabel("Registration cohort")
    ax.set_title("Revenue per cohort user by months since registration")
    ax.spines[:].set_visible(False)
    fig.colorbar(im, ax=ax, label="Revenue per user ($)")
    save(fig, out, "7_cohort_heatmap")


def main():
    ap = argparse.ArgumentParser(description="Create the project charts")
    ap.add_argument("--results", default="results")
    ap.add_argument("--out", default=None, help="default: <results>/figures")
    args, _ = ap.parse_known_args()
    out = args.out or os.path.join(args.results, "figures")
    print(f"reading CSVs from '{args.results}', writing PNGs to '{out}'")
    for fn in (chart_revenue_by_category, chart_monthly_revenue, chart_customer_segments,
               chart_funnel, chart_conversion_by_segment, chart_top_products, chart_cohort_heatmap):
        try:
            fn(args.results, out)
        except Exception as e:           # one bad chart must not stop the others
            print(f"  {fn.__name__} FAILED: {type(e).__name__}: {e}")


if __name__ == "__main__":
    main()


reading CSVs from 'results', writing PNGs to 'results\figures'
  wrote results\figures\1_revenue_by_category.png
  wrote results\figures\2_monthly_revenue.png
  wrote results\figures\3_customer_segments.png
  wrote results\figures\4_conversion_funnel.png
  wrote results\figures\5_conversion_by_segment.png
  wrote results\figures\6_top_products.png
  wrote results\figures\7_cohort_heatmap.png
